<a href="https://colab.research.google.com/github/aadhavjawahar-sys/Poker_hand_predictor/blob/main/Decision_tree_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
#Get neccessary import modules for decision tre
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, classification_report

In [16]:
#Data sets
url_train = "https://raw.githubusercontent.com/aadhavjawahar-sys/Poker_hand_predictor/refs/heads/main/poker-hand-training-true.data"
url_test = "https://raw.githubusercontent.com/aadhavjawahar-sys/Poker_hand_predictor/refs/heads/main/poker-hand-training-true.data"

#Training data manipulation
df = pd.read_csv(url_train)

for i in range(1,6):
  for j in range(i+1,6):
    df["S" + str(i) + "==S" + str(j)]= (df["S" + str(i)] == df["S" + str(j)])
    df["C" + str(i) + "==C" + str(j)]= (df["C" + str(i)] == df["C" + str(j)])
  df.drop(columns="S" + str(i),inplace=True)

#df = df.drop(df.index[100000:]) #TEMPORARY

X_train = df.drop(columns=["CLASS"])
y_train = df["CLASS"]

#Testing data manipulation
df = pd.read_csv(url_test)

for i in range(1,6):
  for j in range(i+1,6):
    df["S" + str(i) + "==S" + str(j)]= (df["S" + str(i)] == df["S" + str(j)])
    df["C" + str(i) + "==C" + str(j)]= (df["C" + str(i)] == df["C" + str(j)])
  df.drop(columns="S" + str(i),inplace=True)

X_test = df.drop(columns=["CLASS"])
y_test = df["CLASS"]

df.head()

,C1,C2,C3,C4,C5,CLASS,S1==S2,C1==C2,S1==S3,C1==C3,...,S2==S4,C2==C4,S2==S5,C2==C5,S3==S4,C3==C4,S3==S5,C3==C5,S4==S5,C4==C5
0,10,11,13,12,1,9,True,False,True,False,...,True,False,True,False,True,False,True,False,True,False
1,11,13,10,12,1,9,True,False,True,False,...,True,False,True,False,True,False,True,False,True,False
2,12,11,13,10,1,9,True,False,True,False,...,True,False,True,False,True,False,True,False,True,False
3,10,11,1,13,12,9,True,False,True,False,...,True,False,True,False,True,False,True,False,True,False
4,1,13,12,11,10,9,True,False,True,False,...,True,False,True,False,True,False,True,False,True,False


In [17]:
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier()
model.fit(X_train, y_train)

DecisionTreeClassifier()

In [18]:
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

In [19]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_train, y_pred_train))
print(classification_report(y_train, y_pred_train))

print(confusion_matrix(y_test, y_pred_test))
print(classification_report(y_test, y_pred_test))

[[12493     0     0     0     0     0     0     0     0     0]
 [    0 10599     0     0     0     0     0     0     0     0]
 [    0     0  1206     0     0     0     0     0     0     0]
 [    0     0     0   513     0     0     0     0     0     0]
 [    0     0     0     0    93     0     0     0     0     0]
 [    0     0     0     0     0    54     0     0     0     0]
 [    0     0     0     0     0     0    36     0     0     0]
 [    0     0     0     0     0     0     0     6     0     0]
 [    0     0     0     0     0     0     0     0     5     0]
 [    0     0     0     0     0     0     0     0     0     5]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     12493
           1       1.00      1.00      1.00     10599
           2       1.00      1.00      1.00      1206
           3       1.00      1.00      1.00       513
           4       1.00      1.00      1.00        93
           5       1.00      1.00      1.00 

In [ ]:
import gradio as gr
import pandas as pd
import numpy as np

# Mappings based on your dataset specifications
SUIT_MAP = {
    "Hearts": 1,
    "Spades": 2,
    "Diamonds": 3,
    "Clubs": 4
}

RANK_MAP = {
    "Ace": 1, "2": 2, "3": 3, "4": 4, "5": 5, "6": 6, "7": 7,
    "8": 8, "9": 9, "10": 10, "Jack": 11, "Queen": 12, "King": 13
}

HAND_CLASSES = {
    0: "Nothing in hand",
    1: "One pair",
    2: "Two pairs",
    3: "Three of a kind",
    4: "Straight",
    5: "Flush",
    6: "Full house",
    7: "Four of a kind",
    8: "Straight flush",
    9: "Royal flush"
}

def predict_poker_hand(s1, c1, s2, c2, s3, c3, s4, c4, s5, c5):
    # Step 0: Check for duplicate cards
    selected_cards = [(s1, c1), (s2, c2), (s3, c3), (s4, c4), (s5, c5)]
    if len(set(selected_cards)) < 5:
        raise gr.Error("Duplicate cards detected! Please select 5 unique cards.")

    # Step 1: Map text selections to numbers
    raw_data = {
        "S1": SUIT_MAP[s1], "C1": RANK_MAP[c1],
        "S2": SUIT_MAP[s2], "C2": RANK_MAP[c2],
        "S3": SUIT_MAP[s3], "C3": RANK_MAP[c3],
        "S4": SUIT_MAP[s4], "C4": RANK_MAP[c4],
        "S5": SUIT_MAP[s5], "C5": RANK_MAP[c5]
    }

    # Create a 1-row DataFrame matching the raw input structure
    df_input = pd.DataFrame([raw_data])

    # Step 2: Apply feature engineering matching training pipeline
    for i in range(1, 6):
        for j in range(i + 1, 6):
            df_input[f"S{i}==S{j}"] = (df_input[f"S{i}"] == df_input[f"S{j}"])
            df_input[f"C{i}==C{j}"] = (df_input[f"C{i}"] == df_input[f"C{j}"])
        # Drop the original suit columns as done in training
        df_input.drop(columns=f"S{i}", inplace=True)

    # Step 3: Predict using pre-existing 'model'
    prediction = model.predict(df_input)

    # Handle scalar vs array predictions
    if hasattr(prediction, "__len__"):
        predicted_class = int(prediction[0])
    else:
        predicted_class = int(prediction)

    return HAND_CLASSES.get(predicted_class, "Unknown Hand")

# Build the Gradio Interface
suits = ["Hearts", "Spades", "Diamonds", "Clubs"]
ranks = ["Ace", "2", "3", "4", "5", "6", "7", "8", "9", "10", "Jack", "Queen", "King"]

# Set default initial values to be unique across cards
default_ranks = ["Ace", "2", "3", "4", "5"]

inputs = []
for i in range(1, 6):
    inputs.append(gr.Dropdown(choices=suits, label=f"Card {i} Suit", value="Hearts"))
    inputs.append(gr.Dropdown(choices=ranks, label=f"Card {i} Rank", value=default_ranks[i - 1]))

interface = gr.Interface(
    fn=predict_poker_hand,
    inputs=inputs,
    outputs=gr.Textbox(label="Predicted Poker Hand"),
    title="Poker Hand Predictor",
    description="Select suit and rank for 5 distinct cards to predict the hand class."
)

# Launch in Colab
interface.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://762d9e6c3e21174764.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
